# QM 640 Capstone — Step 3d: 8-K Item-Code Filter (Objective)

Every 8-K filing declares which numbered "Item" it's reporting under, right
on its cover page — this is metadata the filer checked off, not something
requiring interpretation. Your Synopsis explicitly scopes the study to
**Items 8.01, 1.01, and 2.01 only**.

This notebook pulls each filing's declared item code(s) from SEC's own
submissions API and auto-excludes anything filed under an out-of-scope item
(most commonly Item 2.02 — Results of Operations, i.e. an earnings release
that happened to mention AI in passing, not a genuine investment
announcement). This is purely a metadata check, not a content judgment, so
it's fair game to automate.

**Run this after `03c_automated_flags.ipynb`.**

**v2 note:** this version fixes a bug from the first release where
`NaN or ""` evaluated to `NaN` (Python treats NaN as truthy) instead of
`""`, which corrupted every clean row's `exclude_reason` into the literal
text `"nan"` and made the summary count everything as excluded. Cell 4b
below repairs that if it already happened to your data.

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [1]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "your_email@example.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 915, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 915 (delta 43), reused 67 (delta 25), pack-reused 813 (from 1)
Receiving objects: 100% (915/915), 6.29 MiB | 7.62 MiB/s, done.
Resolving deltas: 100% (484/484), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [2]:
!pip install -q pandas requests

## Cell 3 — Configuration

In [3]:
import os

RAW_DIR = os.path.join(BASE_DIR, "data/raw")
SCREENING_FILE = os.path.join(RAW_DIR, "screening_worksheet.csv")
HEADERS = {"User-Agent": "QM640 Capstone research your_email@example.com"}  # edit to your real email

IN_SCOPE_ITEMS = {"1.01", "2.01", "8.01"}  # per Synopsis Scope Boundaries

## Cell 4 — Load worksheet

In [4]:
import pandas as pd

df = pd.read_csv(SCREENING_FILE)
print(f"Loaded {len(df)} rows")
still_open = df["exclude_reason"].isna() | (df["exclude_reason"].astype(str).str.strip() == "")
print(f"Rows not yet auto-excluded going into this notebook: {still_open.sum()}")

Loaded 11676 rows
Rows not yet auto-excluded going into this notebook: 2179


## Cell 4b — Repair pass (safe to run even if you haven't hit this bug)

An earlier version of this notebook had a bug: `NaN or ""` evaluates to
`NaN` in Python (NaN is truthy, not falsy), not `""`. That turned every
genuinely clean row's blank `exclude_reason` into the literal text `"nan"`,
which then made every row look excluded downstream. This cell repairs that
corruption if present; it's a no-op if your data is already clean.

In [5]:
def clean_reason(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x == "nan":
        return ""
    if x.lower().startswith("nan; "):
        return x[5:]
    return x

before = (df["exclude_reason"].astype(str).str.strip() == "nan").sum()
df["exclude_reason"] = df["exclude_reason"].apply(clean_reason)
print(f"Repaired {before} rows that had the literal string artifact from the old bug")

still_open = df["exclude_reason"].astype(str).str.strip() == ""
print(f"Rows genuinely open (no exclusion reason) after repair: {still_open.sum()}")

Repaired 2179 rows that had the literal string artifact from the old bug
Rows genuinely open (no exclusion reason) after repair: 2179


## Cell 5 — Pull declared item codes per filing (SEC submissions API)

Caches one call per unique CIK, matches by accession number to find each
row's own filing within that company's recent filing history.

In [6]:
import requests
import time

def get_filing_history_with_items(cik):
    cik_padded = str(int(cik)).zfill(10)
    url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json"
    resp = requests.get(url, headers=HEADERS, timeout=20)
    if resp.status_code != 200:
        return None
    data = resp.json()
    recent = data.get("filings", {}).get("recent", {})
    return pd.DataFrame({
        "accessionNumber": recent.get("accessionNumber", []),
        "items": recent.get("items", []),
    })


history_cache = {}
item_codes = []

for _, row in df.iterrows():
    cik = row["cik"]
    if cik not in history_cache:
        history_cache[cik] = get_filing_history_with_items(cik)
        time.sleep(0.15)

    hist = history_cache[cik]
    own_accn = row["accession_no"].split(":")[0] if isinstance(row["accession_no"], str) else None

    if hist is None or hist.empty or own_accn is None:
        item_codes.append(None)
        continue

    match = hist[hist["accessionNumber"] == own_accn]
    item_codes.append(match["items"].iloc[0] if not match.empty else None)

df["item_codes"] = item_codes
print(df["item_codes"].value_counts().head(20))

item_codes
2.02,9.01              3311
7.01,9.01              1839
2.02,7.01,9.01         1277
8.01,9.01              1108
1.01,7.01,9.01          336
5.02,7.01,9.01          320
2.02,8.01,9.01          313
7.01,8.01,9.01          256
1.01,8.01,9.01          166
5.02,9.01               163
1.01,3.02,7.01,9.01     124
1.01,9.01               116
2.02,5.02,9.01          108
2.02,7.01,8.01,9.01      79
1.01,3.02,8.01,9.01      77
9.01                     74
5.02,8.01,9.01           63
2.02,5.02,7.01,9.01      58
7.01                     57
2.02                     55
Name: count, dtype: int64


## Cell 6 — Flag out-of-scope items and update exclude_reason

Fixed: NaN is now handled explicitly with `pd.isna()` instead of the buggy
`x or default` pattern.

In [7]:
def is_in_scope(items_str):
    if pd.isna(items_str) or items_str == "":
        return "REVIEW"  # could not determine - needs a manual glance
    filed_items = {i.strip() for i in str(items_str).split(",")}
    return "Y" if filed_items & IN_SCOPE_ITEMS else "N"


df["item_in_scope"] = df["item_codes"].apply(is_in_scope)


def update_reason(row):
    raw = row.get("exclude_reason", "")
    existing = "" if pd.isna(raw) else str(raw).strip()

    if row["item_in_scope"] == "N":
        addition = f"filed under out-of-scope item(s): {row['item_codes']}"
        return f"{existing}; {addition}" if existing else addition
    return existing


df["exclude_reason"] = df.apply(update_reason, axis=1)
df.to_csv(SCREENING_FILE, index=False)

print(df["item_in_scope"].value_counts())
print(f"\nSaved -> {SCREENING_FILE}")

item_in_scope
N         7681
Y         3908
REVIEW      87
Name: count, dtype: int64

Saved -> /content/QM640-WALSH-CAPSTONE/data/raw/screening_worksheet.csv


## Commit and push results back to GitHub

In [8]:
!git -C {BASE_DIR} add "data/raw/screening_worksheet.csv"
!git -C {BASE_DIR} commit -m "Step 3d: 8-K item-code scope filter (v2, with nan-string bug fix)"
!git -C {BASE_DIR} push

[main b76333c] Step 3d: 8-K item-code scope filter (v2, with nan-string bug fix)
 1 file changed, 11677 insertions(+), 11677 deletions(-)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (5/5), 189.01 KiB | 1.15 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   85dad4f..b76333c  main -> main


## Summary — updated manual-review workload

In [9]:
def is_excluded(x):
    return isinstance(x, str) and len(x.strip()) > 0

auto_excluded = df["exclude_reason"].apply(is_excluded)
needs_review = df["item_in_scope"] == "REVIEW"

print(f"Total candidate filings: {len(df)}")
print(f"Auto-excluded so far (Step 3c + 3d combined): {auto_excluded.sum()}")
print(f"Item code lookup failed, needs a quick manual glance: {needs_review.sum()}")
print(f"Remaining for real manual content review (is_genuine_ai_event / announcement_type): "
      f"{(~auto_excluded).sum()}")

Total candidate filings: 11676
Auto-excluded so far (Step 3c + 3d combined): 10796
Item code lookup failed, needs a quick manual glance: 87
Remaining for real manual content review (is_genuine_ai_event / announcement_type): 880


## Export a small, focused file for the actual manual review

Instead of scrolling through all 2,000+ rows in the master worksheet, this
pulls out just the ones that still need your eyes -- the genuinely open rows
plus the item-code lookup failures -- into a separate, much smaller CSV.

You'll still edit the *master* `screening_worksheet.csv` (matched by
`accession_no`), but this file is what you actually open and work through.

In [10]:
to_review = df[~auto_excluded | needs_review].copy()
review_cols = ["accession_no", "company_name", "file_date", "announcement_type",
               "is_genuine_ai_event", "item_codes", "item_in_scope", "filing_url"]
to_review = to_review[review_cols]

review_path = os.path.join(RAW_DIR, "screening_TO_REVIEW.csv")
to_review.to_csv(review_path, index=False)
print(f"{len(to_review)} rows to review -> {review_path}")
print("\nWorkflow: fill in is_genuine_ai_event / announcement_type in THIS file, "
      "then merge those two columns back into screening_worksheet.csv by accession_no "
      "before running the kappa check (Part B of notebook 03).")

922 rows to review -> /content/QM640-WALSH-CAPSTONE/data/raw/screening_TO_REVIEW.csv

Workflow: fill in is_genuine_ai_event / announcement_type in THIS file, then merge those two columns back into screening_worksheet.csv by accession_no before running the kappa check (Part B of notebook 03).


## Commit and push results back to GitHub

In [11]:
!git -C {BASE_DIR} add "data/raw/screening_TO_REVIEW.csv"
!git -C {BASE_DIR} commit -m "Step 3d: export focused manual-review subset"
!git -C {BASE_DIR} push

[main 087d3a8] Step 3d: export focused manual-review subset
 1 file changed, 923 insertions(+)
 create mode 100644 data/raw/screening_TO_REVIEW.csv
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (5/5), 29.47 KiB | 4.91 MiB/s, done.
Total 5 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   b76333c..087d3a8  main -> main
